In [2]:
import os
iskaggle = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '')
RANDOMSEED=1727

In [ ]:
import torch,random
import tensorflow as tf
import numpy as np
def random_seed(seed=1727, use_cuda=False):
    np.random.seed(seed) # cpu vars
    torch.manual_seed(seed) # cpu vars
    random.seed(seed) # Python
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    if use_cuda: 
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # gpu vars
        torch.backends.cudnn.deterministic = True  #needed
        torch.backends.cudnn.benchmark = False
        
random_seed(1727)
tf.config.list_physical_devices()

In [4]:
from pathlib import Path

cred_path = Path('~/.kaggle/access_token').expanduser()
if not cred_path.exists():
    cred_path.parent.mkdir(exist_ok=True)
    cred_path.write_text("KGAT_9f6b15aaf6f7637b8497dfb3c56c079e")
    cred_path.chmod(0o600)

In [5]:
compName = Path('nlp-getting-started')

In [ ]:
if not iskaggle:
    if not compName.exists():
        import zipfile,kaggle
        kaggle.api.competition_download_cli(str(compName))
        zipfile.ZipFile(f'{compName}.zip').extractall(compName)
else:
    # /kaggle/input/competitions/nlp-getting-started/train.csv
    compName = Path(f'/kaggle/input/competitions/{compName}')

# %pip install -q datasets
!dir /o:g /w {compName}
# !ls {compName}

In [7]:
import pandas as pd
train=pd.read_csv(compName/"train.csv")
test=pd.read_csv(compName/"test.csv")

# drop the `id` col for training data ONLY, we need the id for test preds later
train.drop(columns=["id"],inplace=True)

# feat engineering

In [8]:
train["keywordDefined"]=pd.Series([
  int(pd.notna(keyword)) for keyword in train["keyword"]
])
train["locationDefined"]=pd.Series([
  int(pd.notna(loc)) for loc in train["location"]
])
train["keywordInText"]=pd.Series([
	int(str(keyword).lower() in str(text).lower()) if keyword else 0 for keyword,text in zip(train["keyword"],train["text"])
])
train["has_all_caps"] = (
    train["text"].str.contains(r'\b[A-Z]{3,}\b')
).astype(int)

In [9]:
import re 

def clean_text(text):
    """Clean and preprocess text without NLTK"""
    if pd.isna(text):
        return ""
    
    text = text.lower() # Convert to lowercase
    text = re.sub(r'http\S+|www\S+|https\S+', '', text) # Remove URLs
    text = re.sub(r'@(\w+)', r'\1', text) # Remove mentions (keep the word without @ or #)
    text = re.sub(r'#(\w+)', r'\1', text) # Remove hashtags (keep the word without @ or #)
    text = re.sub(r'\d+', '', text) # Remove numbers
    text = ' '.join(text.split()) # Remove extra whitespace
    
    return text

In [10]:
# Basic text features
train["text_length"] = train["text"].str.len().fillna(0)
train["word_count"] = train["text"].str.split().str.len().fillna(0)

# Additional features (no external libraries)
train["capital_count"] = train["text"].str.findall(r'[A-Z]').str.len()
train["exclamation_count"] = train["text"].str.count('!')
train["question_count"] = train["text"].str.count('\\?')
train["hashtag_count"] = train["text"].str.count('#')
train["mention_count"] = train["text"].str.count('@')

# apply preproc
train["clean_text"] = train["text"].apply(clean_text)

In [ ]:
train

# prep

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp
from sklearn.preprocessing import StandardScaler

# Prepare text features
word_tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)
char_tfidf = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3,5),
    max_features=20000,
    sublinear_tf=True
)
X_train_word = word_tfidf.fit_transform(train["clean_text"])
X_train_char = char_tfidf.fit_transform(train["clean_text"])

# Prepare numeric features
numeric_features = train[[
    "text_length", "word_count", "capital_count", 
    "exclamation_count", "question_count", 
    "hashtag_count", "mention_count", "keywordInText",
    "keywordDefined", "locationDefined", "has_all_caps",
]].fillna(0).values

scaler = StandardScaler()
numeric_features = scaler.fit_transform(numeric_features)

# Combine features
X = sp.hstack([X_train_word, X_train_char, numeric_features])
y = train["target"].values

In [13]:
from sklearn.model_selection import train_test_split, cross_val_score

xtrain,xval,ytrain,yval = train_test_split(
    X, y, test_size=0.2, random_state=RANDOMSEED, stratify=y
)

# training

we'll be ensemblemaxxing for this problem

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, make_scorer
from sklearn.model_selection import GridSearchCV

# =========linear svc==========
print("grid search on linear svc:")
gslsvc=GridSearchCV(
  estimator=CalibratedClassifierCV(
    LinearSVC(max_iter=35000,random_state=RANDOMSEED),
    n_jobs=-1,
    cv=5,
  ),
  param_grid={
    "estimator__C":[0.1, 0.25, 0.5, 0.75, 0.8, 0.95, 1, 1.5, 2, 5, 10, 20, 50],
    "method":["sigmoid","isotonic"],
    
  },
  scoring=make_scorer(f1_score),
  verbose=3,
  n_jobs=-1,
)
gslsvc.fit(xtrain,ytrain)
print(f"Best gslsvc params: {gslsvc.best_params_}\nbest gslsvc f1: {gslsvc.best_score_}")
# =============================

# =========logistic regression==========
print("\ngrid search on logistic regression:")
gslr=GridSearchCV(
	estimator=LogisticRegression(max_iter=35000, random_state=RANDOMSEED),
	param_grid={
		"C": [0.1, 0.25, 0.5, 0.75, 0.8, 0.95, 1, 1.5, 2, 5, 10, 20, 50],
		"solver": ["liblinear", "lbfgs", "saga"],
		"class_weight": [None, "balanced"],
	},
	scoring=make_scorer(f1_score),
  	verbose=3,
    n_jobs=-1,
)
gslr.fit(xtrain,ytrain)
print(f"Best gslr params: {gslr.best_params_}\nbest gslr f1: {gslr.best_score_}")
# ======================================

In [ ]:
model = VotingClassifier(
    estimators=[
        ('lr', gslr.best_estimator_),
        ('lsvc', gslsvc.best_estimator_),
    ],
    voting='soft' 
)

model.fit(xtrain,ytrain)

In [ ]:
probs = model.predict_proba(xval)[:,1]
best_f1 = 0
best_thresh = 0.5

for t in np.arange(0.1, 0.9, 0.01):
    preds = (probs > t).astype(int)
    f1 = f1_score(yval, preds)

    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print(f"best threshold: {best_thresh}\nbest f1: {best_f1}")

In [ ]:
model.fit(X,y) # fit on full X & y 

# predictions

feat engineering for test data

In [18]:
test["keywordDefined"]=pd.Series([
  int(pd.notna(keyword)) for keyword in test["keyword"]
])
test["locationDefined"]=pd.Series([
  int(pd.notna(loc)) for loc in test["location"]
])
test["keywordInText"]=pd.Series([
	int(str(keyword).lower() in str(text).lower()) if keyword else 0 for keyword,text in zip(test["keyword"],test["text"])
])
test["has_all_caps"] = (
    test["text"].str.contains(r'\b[A-Z]{3,}\b')
).astype(int)

test["clean_text"] = test["text"].apply(clean_text)

test["text_length"] = test["text"].str.len().fillna(0)
test["word_count"] = test["text"].str.split().str.len().fillna(0)
test["capital_count"] = test["text"].str.findall(r'[A-Z]').str.len()
test["exclamation_count"] = test["text"].str.count('!')
test["question_count"] = test["text"].str.count('\\?')
test["hashtag_count"] = test["text"].str.count('#')
test["mention_count"] = test["text"].str.count('@')

prep

In [19]:
X_test_word = word_tfidf.transform(test["clean_text"])
X_test_char = char_tfidf.transform(test["clean_text"])

test_numeric = test[[
    "text_length", "word_count", "capital_count", 
    "exclamation_count", "question_count", 
    "hashtag_count", "mention_count", "keywordInText",
    "keywordDefined", "locationDefined", "has_all_caps",
]].fillna(0).values
test_numeric = scaler.transform(test_numeric)

xtest = sp.hstack([X_test_word, X_test_char, test_numeric])

predict

In [20]:
preds = model.predict_proba(xtest)[:,1]
preds = (preds > best_thresh).astype(int)

In [ ]:
print(f"Unique prediction values: {np.unique(preds)}")

In [22]:
subs_df=pd.DataFrame({
  "ID":test["id"],
  "Target":preds,
})

In [ ]:
# subs_df.head()
subs_df.shape

In [24]:
subs_df.to_csv("submission.csv",index=False)